# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library for machine learning-ready data loading from Croissant schemas.

### Dataset Source
The dataset source is provided as a Croissant schema URL:
*URL*: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install mlcroissant matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata (schema and utility)
ds = mlc.Dataset(croissant_url)
meta = ds.metadata

print(f"Dataset name: {getattr(meta, 'name', None)}\nDescription: {getattr(meta, 'description', None)}\nVersion: {getattr(meta, 'version', None)}\nConforms to: {getattr(meta, 'conformsTo', None)}")

## 2. Data Overview
List available record sets, their fields, and associated `@id`s.

A RecordSet represents a logical table of records (data entities) described by fields/columns. We reference entities by their Croissant `@id`.

In [ ]:
# List all record sets and their fields, referencing by @id
record_sets = getattr(meta, 'recordSet', [])
if not record_sets:
    record_sets = []  # Some Croissant schemas use 'recordSets'

def reduce_to_id_list(entity_or_list, id_key='@id'):
    if isinstance(entity_or_list, list):
        return [item.get('@id', None) if isinstance(item, dict) and '@id' in item else item for item in entity_or_list]
    elif isinstance(entity_or_list, dict):
        if '@id' in entity_or_list:
            return [entity_or_list['@id']]
        return [entity_or_list]
    else:
        return []

print(f"Number of record sets: {len(record_sets)}")
fields_per_record_set = {}
for rec_set in record_sets:
    rec_id = rec_set.get('@id', None)
    print(f"RecordSet @id: {rec_id}, name: {rec_set.get('name', None)}")
    # Each record set has a 'field' key listing the field objects (@ids)
    field_list = rec_set.get('field', [])
    field_ids = reduce_to_id_list(field_list)
    fields_per_record_set[rec_id] = field_ids
    print(f"  Fields: {field_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Let's select the main data table for protein entries. Most FAIR^2 mass-spectrometry datasets provide a primary table with an `@id` typically containing "recordSet". Find the main data record set in the listing above.

In [ ]:
# Identify main tabular data record set (uses @id field with 'recordSet' or similar)
main_record_set_id = None
for rec_set in getattr(meta, 'recordSet', []):
    rec_id = rec_set.get('@id')
    # Heuristic: choose the record set that is likely main by field coverage or by a keyword
    field_ids = rec_set.get('field', [])
    field_count = len(field_ids) if isinstance(field_ids, list) else 1
    if (not main_record_set_id or field_count > len(fields_per_record_set.get(main_record_set_id, []))):
        main_record_set_id = rec_id
# If you know the @id exactly, you can set main_record_set_id = '<@id>'

if not main_record_set_id:
    raise Exception("Could not identify a main record set id.")
print(f"Main record set ID: {main_record_set_id}")

# If there are other side tables or record sets, include them as well:
record_set_ids = [main_record_set_id]

# Load data from the main record set into a pandas DataFrame
dataframes = {}
for rec_set_id in record_set_ids:
    try:
        df = pd.DataFrame(ds.records(record_set=rec_set_id))
    except Exception as e:
        df = pd.DataFrame()
        print(f"Failed to load data for {rec_set_id}: {e}")
    dataframes[rec_set_id] = df

print(f"Loaded columns in main table: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We now explore the main record set. We'll select a numeric column (such as 'molecular weight', 'peptide count', or a normalized abundance) using its Croissant Field `@id`. Replace variable assignments below with the desired field `@id` found in the output above.

In [ ]:
# --- Fields ---
# Please set these according to your dataset's fields and their @id.
# Example placeholder IDs (replace with real ones if known):
# See previous cell for actual column names.
numeric_field_id = None
for col in dataframes[main_record_set_id].columns:
    # Heuristic: pick a likely numeric field, like 'mw', 'molecular_weight', or abundance
    col_lc = str(col).lower()
    if any(s in col_lc for s in ['mw', 'molecular', 'mass', 'weight', 'abundance', 'peptide_count', 'count']):
        numeric_field_id = col
        break
if not numeric_field_id:
    # fallback: try the first floating-point column
    for col in dataframes[main_record_set_id].select_dtypes(include=['float', 'int']).columns:
        numeric_field_id = col
        break
if not numeric_field_id:
    raise Exception('No numeric field found for EDA.')
print(f"Numeric field chosen for EDA: {numeric_field_id}")

df = dataframes[main_record_set_id].copy()
# Remove non-numeric, coercing errors, and drop NaNs
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
numeric_df = df.dropna(subset=[numeric_field_id])

# Example: Remove outliers (e.g., 99th percentile), filter field, normalize, group
upper = numeric_df[numeric_field_id].quantile(0.99)
lower = numeric_df[numeric_field_id].quantile(0.01)
filtered_df = numeric_df[(numeric_df[numeric_field_id] >= lower) & (numeric_df[numeric_field_id] <= upper)]
print(f"Filtered to central 98% of {numeric_field_id} values: {filtered_df.shape[0]} records")

# Normalize column (Z-score)
filtered_df[f"{numeric_field_id}_zscore"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_zscore"]].head())

# Example: group by a categorical field (e.g., 'description', 'accession', etc) if present
group_field_id = None
for col in filtered_df.columns:
    col_lc = str(col).lower()
    # Choose a sensible group like protein type, accession, sample type, etc.
    if any(s in col_lc for s in ['type', 'group', 'category', 'sample', 'accession', 'protein']):
        if col != numeric_field_id:
            group_field_id = col
            break

if group_field_id:
    grouped = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    )
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Here we visualize the distribution of the selected numeric field and, if applicable, by category/group.

In [ ]:
# Distribution of the numeric field
plt.figure(figsize=(8,5))
plt.hist(filtered_df[numeric_field_id], bins=30, alpha=0.7, color='steelblue')
plt.title(f'Histogram: {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field found, boxplot by group
if group_field_id:
    # Take top N groups for clarity
    top_groups = filtered_df[group_field_id].value_counts().head(6).index
    subset = filtered_df[filtered_df[group_field_id].isin(top_groups)]
    plt.figure(figsize=(10,5))
    subset.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id} (top groups)')
    plt.suptitle("")
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
- We loaded the FAIR^2 dataset Croissant schema and extracted its main data table via its `@id`.
- Explored numerical fields (such as molecular weight, abundance, or peptide count) and visualized their distribution.
- Demonstrated field normalization (z-scoring) and basic grouping.

For deeper analysis, refer to the Croissant schema online or the [mlcroissant documentation](https://github.com/mlcommons/croissant). Adjust field `@id`s and analysis code to match your scientific question and the table structure.